In [ ]:
import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import AdamW
import matplotlib.pyplot as plt  # 追加

class SpeedEstimationModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(15, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 + 15 + 15, 128),
            nn.LeakyReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, imgs, speeds, distances):
        x = imgs.squeeze(2)
        x = self.cnn(x)
        x = x.view(x.size(0), -1)
        x = torch.cat([x, speeds, distances], dim=1)
        return self.fc(x).squeeze(1)

class SpeedDataset(Dataset):
    def __init__(self, crop_root, annot_root, distance_json_path=None, max_items=150):
        self.items = []
        self.transform = transforms.Compose([
            transforms.Resize((64, 64)),
            transforms.ToTensor()
        ])
        self.distances = {}
        if distance_json_path:
            with open(distance_json_path, encoding='utf-8') as f:
                self.distances = json.load(f)

        scene_ids = sorted(os.listdir(crop_root))
        for sid in scene_ids:
            if int(sid) > 240: continue
            crop_dir = os.path.join(crop_root, sid)
            annot_path = os.path.join(annot_root, f"{sid}.json")
            if not os.path.exists(annot_path): continue
            files = sorted([f for f in os.listdir(crop_dir) if f.endswith(".png")])
            if len(files) == 0: continue
            with open(annot_path, encoding='utf-8') as f:
                ann = json.load(f)
            min_len = min(len(files), len(ann['sequence']))
            if min_len < 15: continue

            speeds_all = np.array([ann['sequence'][i]['OwnSpeed'] / 3.6 for i in range(min_len)], dtype=np.float32)
            tgt_speeds_all = np.array([ann['sequence'][i]['TgtSpeed_ref'] / 3.6 for i in range(min_len)], dtype=np.float32)
            rel_speeds_all = tgt_speeds_all - speeds_all

            val = self.distances.get(str(sid), [])
            if isinstance(val, dict):
                val = list(val.values())
            distances_all = val
            if len(distances_all) < min_len:
                distances_all += [distances_all[-1] if distances_all else 20.0] * (min_len - len(distances_all))
            distances_all = np.array(distances_all[:min_len], dtype=np.float32)

            used_indices = set()

            for i in range(min_len - 14):
                if len(self.items) >= max_items:
                    return
                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                speeds = speeds_all[i:i+15]
                distances = distances_all[i:i+15]
                tgt_rel_speed = np.mean(rel_speeds_all[i:i+15])
                if len(speeds) < 15 or len(distances) < 15:
                    continue
                self.items.append((img_paths, speeds, distances, tgt_rel_speed, sid))
                used_indices.add(i)

            if (min_len - 15) not in used_indices:
                i = min_len - 15
                img_paths = [os.path.join(crop_dir, files[j]) for j in range(i, i + 15)]
                speeds = speeds_all[i:i+15]
                distances = distances_all[i:i+15]
                tgt_rel_speed = np.mean(rel_speeds_all[i:i+15])
                if len(speeds) == 15 and len(distances) == 15:
                    self.items.append((img_paths, speeds, distances, tgt_rel_speed, sid))

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        img_paths, speeds, distances, tgt_rel_speed, scene_id = self.items[idx]
        imgs = []
        for p in img_paths:
            try:
                img = Image.open(p).convert("L")
                imgs.append(self.transform(img))
            except:
                imgs.append(torch.zeros((1, 64, 64)))
        imgs = torch.stack(imgs)
        return imgs, torch.tensor(speeds), torch.tensor(distances), torch.tensor(tgt_rel_speed), scene_id

def collate_fn(batch):
    imgs, speeds, distances, tgts, sids = zip(*batch)
    return torch.stack(imgs), torch.stack(speeds), torch.stack(distances), torch.tensor(tgts), list(sids)

def train_speed_model2():
    crop_dir = "disparity_crops"
    annot_dir = "train_annotations"
    distance_path = "./outputs/distance_estimates.json"

    dataset = SpeedDataset(crop_dir, annot_dir, distance_path, max_items=150)
    train_ids, val_ids = train_test_split(list(range(len(dataset))), test_size=0.2, random_state=42)
    train_ds = torch.utils.data.Subset(dataset, train_ids)
    val_ds = torch.utils.data.Subset(dataset, val_ids)

    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SpeedEstimationModel2().to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []

    for epoch in range(250):
        model.train()
        total_train_loss = 0
        for imgs, speeds, distances, tgts, _ in tqdm(train_loader, desc=f"[Train Epoch {epoch+1}]"):
            imgs, speeds, distances, tgts = imgs.to(device), speeds.to(device), distances.to(device), tgts.to(device)
            pred = model(imgs, speeds, distances)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * imgs.size(0)

        avg_train_loss = total_train_loss / len(train_ds)
        train_losses.append(avg_train_loss)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for imgs, speeds, distances, tgts, _ in val_loader:
                imgs, speeds, distances, tgts = imgs.to(device), speeds.to(device), distances.to(device), tgts.to(device)
                pred = model(imgs, speeds, distances)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * imgs.size(0)

        avg_val_loss = total_val_loss / len(val_ds)
        val_losses.append(avg_val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    torch.save(model.state_dict(), "speed_model2_rel_withdist.pth")
    print("speed_model2_rel_withdist.pthを保存")

    # グラフ描画
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss over Epochs')
    plt.legend()
    plt.grid(True)
    plt.savefig("loss_curve.png")
    plt.show()

if __name__ == "__main__":
    train_speed_model2()
